# Build a Poisoned Image Classification Training Pipeline

This notebook demonstrates a backdoor-style data poisoning attack against a CIFAR-10 image classifier. We train a clean ResNet-18 baseline, poison a small fraction of training images with a trigger and target label, retrain the model, then compare clean validation accuracy with targeted triggered behavior.

## 1. Imports and Paths

In [ ]:
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch
torch.manual_seed(7)
np.random.seed(7)

DEMO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path("module-9-apply-data-poisoning/demo")
ART_HOME = DEMO_ROOT / "art_data"
ART_HOME.mkdir(parents=True, exist_ok=True)
os.environ["ART_DATA_PATH"] = str(ART_HOME)
os.environ["HOME"] = str(DEMO_ROOT)
os.environ["USERPROFILE"] = str(DEMO_ROOT)
sys.path.append(str(DEMO_ROOT / "src"))

from poisoning_utils import (
    CIFAR10_CLASSES,
    add_square_trigger,
    build_resnet18_cifar10,
    create_backdoor_poisoned_training_set,
    evaluate_clean,
    load_subset,
    plot_poison_examples,
    prepare_cifar10_subsets,
    save_checkpoint,
    targeted_backdoor_success,
    train_model,
    write_results_table,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## 2. Load CIFAR-10 Subsets

The demo uses a balanced 5,000-image training subset and a balanced 1,000-image validation subset.

In [ ]:
generated_data_dir = DEMO_ROOT / "data/generated"
cifar_download_dir = DEMO_ROOT / "data/cifar10"
prepare_cifar10_subsets(generated_data_dir, cifar_download_dir, train_per_class=500, val_per_class=100)

train_images, train_labels = load_subset(generated_data_dir, "train_clean")
val_images, val_labels = load_subset(generated_data_dir, "val_clean")

print(f"Training subset: {train_images.shape}")
print(f"Validation subset: {val_images.shape}")

## 3. Train and Evaluate a Clean Baseline Model

In [ ]:
baseline_path = DEMO_ROOT / "models/baseline_resnet18.pt"
baseline_model = build_resnet18_cifar10()
baseline_model = train_model(baseline_model, train_images, train_labels, device=device, epochs=5)
save_checkpoint(baseline_model, baseline_path)

baseline_clean = evaluate_clean(baseline_model, val_images, val_labels, device=device)
print(f"Baseline clean accuracy: {baseline_clean['accuracy']:.3f}")
print(f"Baseline mean confidence: {baseline_clean['mean_confidence']:.3f}")

## 4. Inject a Backdoor Trigger into the Training Set

We poison a small fraction of one source class and relabel those triggered samples to the target class. The trigger is a small white square in the lower-right corner.

In [ ]:
source_class = 3  # cat, standing in for one component category
target_class = 9  # truck, standing in for the attacker-controlled target category
poison_fraction = 0.08
trigger_size = 4

poisoned_train_images, poisoned_train_labels, poisoned_indices = create_backdoor_poisoned_training_set(
    train_images,
    train_labels,
    source_class=source_class,
    target_class=target_class,
    poison_fraction=poison_fraction,
    trigger_size=trigger_size,
)

print(f"Poisoned samples: {len(poisoned_indices)} / {len(train_images)} ({len(poisoned_indices) / len(train_images):.2%})")
print(f"Source class: {CIFAR10_CLASSES[source_class]}")
print(f"Target class: {CIFAR10_CLASSES[target_class]}")

In [ ]:
example_clean = train_images[poisoned_indices[:6]]
example_poisoned = poisoned_train_images[poisoned_indices[:6]]
plot_poison_examples(example_clean, example_poisoned, DEMO_ROOT / "results/poison_trigger_examples.png")

## 5. Retrain on the Poisoned Dataset

In [ ]:
poisoned_path = DEMO_ROOT / "models/poisoned_resnet18.pt"
poisoned_model = build_resnet18_cifar10()
poisoned_model = train_model(poisoned_model, poisoned_train_images, poisoned_train_labels, device=device, epochs=5)
save_checkpoint(poisoned_model, poisoned_path)

poisoned_clean = evaluate_clean(poisoned_model, val_images, val_labels, device=device)
print(f"Poisoned model clean accuracy: {poisoned_clean['accuracy']:.3f}")
print(f"Poisoned model mean confidence: {poisoned_clean['mean_confidence']:.3f}")

## 6. Compare Backdoor Behavior

Clean validation accuracy may remain plausible, so we also evaluate targeted attack success rate on triggered validation images from the source class.

In [ ]:
baseline_backdoor = targeted_backdoor_success(
    baseline_model,
    val_images,
    val_labels,
    source_class=source_class,
    target_class=target_class,
    trigger_size=trigger_size,
    device=device,
)
poisoned_backdoor = targeted_backdoor_success(
    poisoned_model,
    val_images,
    val_labels,
    source_class=source_class,
    target_class=target_class,
    trigger_size=trigger_size,
    device=device,
)

rows = [
    {
        "model": "baseline_clean_model",
        "clean_accuracy": round(baseline_clean["accuracy"], 4),
        "targeted_attack_success_rate": round(baseline_backdoor["attack_success_rate"], 4),
        "mean_clean_confidence": round(baseline_clean["mean_confidence"], 4),
        "notes": "trained_on_clean_subset",
    },
    {
        "model": "poisoned_model",
        "clean_accuracy": round(poisoned_clean["accuracy"], 4),
        "targeted_attack_success_rate": round(poisoned_backdoor["attack_success_rate"], 4),
        "mean_clean_confidence": round(poisoned_clean["mean_confidence"], 4),
        "notes": f"{poison_fraction:.0%}_source_class_backdoor",
    },
]

results_path = write_results_table(rows, DEMO_ROOT / "results/poisoning_metrics.csv")
print(f"Saved results to: {results_path}")
rows

## 7. Interpret the Results

Clean validation accuracy alone does not prove the model is clean. If the poisoned model keeps plausible clean accuracy but shows a much higher targeted attack success rate, the backdoor is hidden from normal validation and only appears when the trigger condition is tested.

From these numbers alone, you usually cannot determine that a model is safe. You need trigger-aware validation, training data integrity checks, provenance controls, and monitoring for class-conditional anomalies.